In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.mbe import patch_mbe 
import torch 
from src.gapt import GatedPhaseTransition

# ----- load model -----
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ----- initialize GAPT ----- 
gapt = GatedPhaseTransition(
    p_m=125, # patience plateaued ce loss 
    p_a=75 # patience for plateaued mbe loss 
)

# ----- compute loss ----
num_layers = model.config.num_hidden_layers
per_layer_mbe_mask = torch.zeros(num_layers) # <- require ablation on this mask
per_layer_mbe_mask[1:-1] = 1 
patch_size = 3

inputs = tokenizer("Hello, how are you?", return_tensors="pt")
assert len(inputs["input_ids"][0]) % patch_size == 0, "patch size needs to divide seq len here"

# Request #1. Include 'spike' & 'bottleneck' schedule to replace the 'naive' schedule here
def compute_loss(model, inputs, patch_size: int, per_layer_mbe_mask: torch.Tensor, gapt: GatedPhaseTransition, mbe_comp_mode: str = "spike"):
    outputs = model(**inputs, labels=inputs["input_ids"], output_hidden_states=True, return_dict=True)
    ce_loss = outputs.loss 

    mbe_per_layer = torch.stack([patch_mbe(h, patch_size).float() for h in outputs.hidden_states[1:]])
    masked_mbe = mbe_per_layer * per_layer_mbe_mask
    if mbe_comp_mode == "naive": 
        mbe_loss = (masked_mbe.sum() / per_layer_mbe_mask.sum())
    elif mbe_comp_mode == "bottleneck": 
        gradients = masked_mbe[1:] - masked_mbe[:-1]
        k = max(1, int(len(masked_mbe) * 0.1))
        _, indices = torch.topk(gradients, k, largest=False)
        mbe_loss = masked_mbe[indices + 1].mean()
    elif mbe_comp_mode == "spike": 
        gradients = masked_mbe[1:] - masked_mbe[:-1]
        decay_idx = gradients.argmin()
        mbe_loss = masked_mbe[decay_idx + 1]

    loss = gapt.step(ce_loss, mbe_loss)
    return loss

In [2]:
from src.gapt_trainer import GaptTrainer, GaptConfig
from transformers import TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# ----- load model -----
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

dataset = load_dataset("gsm8k", "main")

# Formatting function
def format_gsm8k(example):
    return {"text": example["question"] + "\n" + example["answer"]}

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

tokenized_datasets = dataset.map(format_gsm8k).map(tokenize_function, batched=True)


# ---- GAPT config & trainer ----
gapt_config = GaptConfig()
gapt_trainer = GaptTrainer(
    gapt_config = gapt_config,
    model = model,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    args = TrainingArguments(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        num_train_epochs=1,
    )
)

# train 
gapt_trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/ksgk/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.514500
1000,0.482900
1500,0.448600
2000,0.444600
2500,0.426700
3000,0.417400
3500,0.413500
4000,0.409700
4500,0.382500
5000,0.384700


TrainOutput(global_step=7473, training_loss=0.40968247409423925, metrics={'train_runtime': 9353.3669, 'train_samples_per_second': 0.799, 'train_steps_per_second': 0.799, 'total_flos': 1.0111836053569536e+16, 'train_loss': 0.40968247409423925, 'epoch': 1.0})

In [ ]:
python data/codeparrot.py 
python data/openweb_math.py
python data/textbook.py
python data/

In [ ]:
# upload raw .pt to huggingface repo
# download raw .pt from huggingface repo

local_ckpt_path = "ckpt/fineweb10B-gpt2-small-20k.pt"


In [ ]:
# gpt2 small spike: layer 5 bottleneck, layer 10 next
# gpt2 medium spike: layer 11 bottleneck, layer 15 next
# gpt2 large spike (softplus): layer 3, next layer 16 & layer 2
# gpt2 xl spike: layer 22 next layer 23

In [6]:
# --- TBD: download from huggingface repo, then run train_iblm after pruning b_layers ---
iblm_gpt = {
    "small": {"hf_repo": "Ksgk-fy/iblm-gpt2-ckpt", "remote_filename": "fineweb10B-iblm-gpt2-small-spike.pt", "b_layer": [5, 10]},
    "medium": {"hf_repo": "Ksgk-fy/iblm-gpt2-ckpt", "remote_filename": "fineweb10B-iblm-gpt2-medium-spike.pt", "b_layer": [11, 15]},
    "large": {"hf_repo": "Ksgk-fy/iblm-gpt2-ckpt", "remote_filename": "fineweb10B-iblm-gpt2-large-softplus-spike.pt", "b_layer": [3, 16, 2]},
    "xl": {"hf_repo": "Ksgk-fy/iblm-gpt2-ckpt", "remote_filename": "fineweb10B-iblm-gpt2-xl-spike.pt", "b_layer": [22, 23]}
}

In [4]:
from huggingface_hub import HfApi, login

# 1. Login (if not already logged in via CLI)
# login(token="YOUR_HF_TOKEN") 

# 2. Configuration
hf_username = "Ksgk-fy"
repo_name = "iblm-gpt2-ckpt" # or whatever repo name you prefer
full_repo_id = f"{hf_username}/{repo_name}"
local_ckpt_path = "ckpt/fineweb10B-gpt2-large-softplus-spike.pt"
remote_filename = "fineweb10B-iblm-gpt2-large-softplus-spike.pt" # Name it will have in the repo

# 3. Create Repo (if it doesn't exist)
api = HfApi()
try:
    api.create_repo(repo_id=full_repo_id, repo_type="model", exist_ok=True)
    print(f"Repo {full_repo_id} ready.")
except Exception as e:
    print(f"Error creating repo (might exist): {e}")

# 4. Upload File
print(f"Uploading {local_ckpt_path} to {full_repo_id}...")
api.upload_file(
    path_or_fileobj=local_ckpt_path,
    path_in_repo=remote_filename,
    repo_id=full_repo_id,
    repo_type="model",
    commit_message=f"Upload raw checkpoint: {remote_filename}"
)

print(f"Upload complete! View at: https://huggingface.co/{full_repo_id}/blob/main/{remote_filename}")

Repo Ksgk-fy/iblm-gpt2-ckpt ready.
Uploading ckpt/fineweb10B-gpt2-large-softplus-spike.pt to Ksgk-fy/iblm-gpt2-ckpt...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete! View at: https://huggingface.co/Ksgk-fy/iblm-gpt2-ckpt/blob/main/fineweb10B-iblm-gpt2-large-softplus-spike.pt


In [ ]:
# load in tinystories validation dataset
